# Notebook 06a — Residualizacion de VitD y Calcio por finca
### Tuberculosis bovina · Target: Lesiones_TB

**Pregunta:** Un animal con *mas VitD que la media de su propia finca* ¿tiene menor
probabilidad de TB? ¿Existe un efecto individual de VitD/Calcio una vez que se elimina
la senal de "a que finca pertenece"?

**Estrategia — residualizacion intra-finca:**

```
VitD_res   = VitD   - media(VitD   | finca)
Calcio_res = Calcio - media(Calcio | finca)
```

Los residuales son ortogonales a la media de finca: el RF no puede usar la finca
de forma implicita a traves de VitD/Calcio.
Se conserva toda la N (n=103 tras `drop_sparse_rows`).

**Preprocesamiento:** identico al Nb 02c.
- `drop_sparse_rows`: elimina filas con >4 NaN en MODEL_FEATURES.
- Los 1-2 NaN restantes se imputan (mediana) dentro del pipeline CV.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             brier_score_loss, matthews_corrcoef, recall_score)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
import shap
import tb_utils as tb

PALETTE = tb.set_plot_style()
tb.set_seeds(tb.SEED)

DATA = "../BD.csv"
df   = tb.clean(tb.load_raw(DATA))

# --- Identico al Nb 02c ---
d = tb.target_subset(df, "Lesiones_TB")
d = tb.drop_sparse_rows(d)          # elimina filas con >4 NaN; n=103
y = d["Lesiones_TB"].astype(int).values
g = d[tb.GROUP_COL].values
EXPLOTS = sorted(d[tb.GROUP_COL].unique())
EXPL_PAL = dict(zip(EXPLOTS, PALETTE[:len(EXPLOTS)]))

print(f"n={len(y)}, prevalencia={y.mean():.3f}")
print(f"Explotaciones: {EXPLOTS}")


## 2. Residualizacion intra-finca de VitD y Calcio

In [ ]:
# Calcular medias de finca (nanmean, ignorando NaN) y residuales
for feat in ["VITAMINA_D", "CALCIO"]:
    farm_mean = d.groupby(tb.GROUP_COL)[feat].transform("mean")
    d[feat + "_res"] = d[feat] - farm_mean

# Descriptivos de medias de finca y residuales
print("Medias de finca:")
print(d.groupby(tb.GROUP_COL)[["VITAMINA_D", "CALCIO"]].mean().round(3))
print()

print("Residuales — estadisticos globales (media ~0):")
for feat_res in ["VITAMINA_D_res", "CALCIO_res"]:
    mu  = round(float(d[feat_res].mean()), 4)
    lo  = round(float(d[feat_res].min()),  2)
    hi  = round(float(d[feat_res].max()),  2)
    nan = int(d[feat_res].isna().sum())
    print(f"  {feat_res}: media={mu}, rango=[{lo}, {hi}], NaN={nan}")

print()
print("Media de residuales por finca (debe ser ~0 en cada una):")
for feat_res in ["VITAMINA_D_res", "CALCIO_res"]:
    fm = d.groupby(tb.GROUP_COL)[feat_res].mean().round(4)
    print(f"  {feat_res}: {fm.to_dict()}")

print()
print("NaN en residuales son heredados de NaN en la variable original.")
print("Se imputaran con mediana dentro del pipeline CV (igual que en 02c).")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

pairs = [("VITAMINA_D", "VITAMINA_D_res", "Vitamina D"),
         ("CALCIO",     "CALCIO_res",     "Calcio")]

for row_i, (feat_raw, feat_res, pretty) in enumerate(pairs):
    # Variable cruda
    ax = axes[row_i][0]
    sns.boxplot(data=d, x=tb.GROUP_COL, y=feat_raw, order=EXPLOTS,
                palette=EXPL_PAL, ax=ax)
    sns.stripplot(data=d, x=tb.GROUP_COL, y=feat_raw, order=EXPLOTS,
                  color="black", alpha=0.3, size=3, jitter=True, ax=ax)
    ax.set_title(f"{pretty} cruda (incluye efecto de finca)")
    ax.set_xlabel("Explotacion"); ax.set_ylabel(feat_raw)

    # Residual intra-finca, coloreado por estado TB
    ax2 = axes[row_i][1]
    hue_vals = d["Lesiones_TB"].map({0: "TB-", 1: "TB+"})
    sns.boxplot(data=d, x=tb.GROUP_COL, y=feat_res, order=EXPLOTS,
                palette=EXPL_PAL, ax=ax2, boxprops=dict(alpha=0.4))
    sns.stripplot(data=d, x=tb.GROUP_COL, y=feat_res, order=EXPLOTS,
                  hue=hue_vals,
                  palette={"TB-": PALETTE[2], "TB+": PALETTE[0]},
                  alpha=0.65, size=4, jitter=True, ax=ax2)
    ax2.axhline(0, color="black", lw=0.9, ls="--")
    ax2.set_title(f"{pretty} residual intra-finca
(puntos: TB+/TB-)")
    ax2.set_xlabel("Explotacion"); ax2.set_ylabel(feat_res)
    if row_i == 0:
        ax2.legend(title="Estado TB", fontsize=8, loc="upper right")
    else:
        ax2.get_legend().remove() if ax2.get_legend() else None

plt.suptitle("VitD y Calcio: crudos vs residuales intra-finca", fontsize=12)
plt.tight_layout()
plt.savefig("figures/fig_06a_residuales.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Random Forest con residuales intra-finca (CV por animal, 5x10)

Se sustituyen VITAMINA_D y CALCIO por sus residuales. El resto de features
(EDAD, serologia, RAZA2) se mantienen sin cambio.
La variable de explotacion **no entra en el modelo en ninguna forma**.


In [ ]:
# Features del modelo residualizado
MODEL_RES = [
    "VITAMINA_D_res", "CALCIO_res",
    "PIROPLASMA_Q_log", "EDAD", "PIROPLASMA", "THEILERIA", "ANAPLASMA", "RAZA2",
]
NUM_RES = ["VITAMINA_D_res", "CALCIO_res",
           "PIROPLASMA_Q_log", "EDAD", "PIROPLASMA", "THEILERIA", "ANAPLASMA"]
CAT_RES = ["RAZA2"]

X_res = d[MODEL_RES].copy()

def make_rf(n_estimators=100, min_samples_leaf=5):
    return RandomForestClassifier(
        n_estimators=n_estimators, max_depth=None,
        min_samples_leaf=min_samples_leaf, max_features="sqrt",
        class_weight="balanced", random_state=tb.SEED, n_jobs=-1, oob_score=True,
    )

def make_pipe_res(feature_list, n_estimators=100):
    num_f = [f for f in feature_list if f in NUM_RES]
    cat_f = [f for f in feature_list if f in CAT_RES]
    prep  = tb.make_preprocessor(numeric=num_f, categorical=cat_f, scale=False)
    return Pipeline([("prep", prep), ("clf", make_rf(n_estimators=n_estimators))])

CV_OUTER = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=tb.SEED)
CV_SEL   = RepeatedStratifiedKFold(n_splits=5, n_repeats=3,  random_state=tb.SEED)

def cv_metrics(feature_list, X, y, cv, n_estimators=100, detailed=False):
    Xs = X[list(feature_list)]
    pipe = make_pipe_res(feature_list, n_estimators)
    rows = []
    for tr, te in cv.split(Xs, y):
        if len(np.unique(y[te])) < 2: continue
        pf = clone(pipe); pf.fit(Xs.iloc[tr], y[tr])
        p  = pf.predict_proba(Xs.iloc[te])[:, 1]
        pr = (p >= 0.5).astype(int)
        rows.append(dict(
            prauc=average_precision_score(y[te], p),
            roc=roc_auc_score(y[te], p),
            brier=brier_score_loss(y[te], p),
            mcc=matthews_corrcoef(y[te], pr),
            sens=recall_score(y[te], pr, pos_label=1, zero_division=0),
            spec=recall_score(y[te], pr, pos_label=0, zero_division=0),
        ))
    df_r = pd.DataFrame(rows)
    return df_r if detailed else (df_r.prauc.mean(), df_r.prauc.std())

base_m, base_s = cv_metrics(MODEL_RES, X_res, y, CV_SEL)
print(f"Baseline RF residualizado: PR-AUC = {base_m:.3f} +/- {base_s:.3f}")
print(f"Linea base prevalencia: {y.mean():.3f}")


In [ ]:
# Importancia de permutacion -> orden de eliminacion
full_pipe = make_pipe_res(MODEL_RES).fit(X_res, y)
pi = permutation_importance(full_pipe, X_res, y, scoring="average_precision",
                             n_repeats=50, random_state=tb.SEED, n_jobs=-1)
imp_df = pd.DataFrame({
    "feature": MODEL_RES,
    "imp_mean": pi.importances_mean,
    "imp_std":  pi.importances_std,
}).sort_values("imp_mean", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 4))
order = imp_df.sort_values("imp_mean")
ax.barh(order.feature, order.imp_mean, xerr=order.imp_std,
        color=PALETTE[0], capsize=3, alpha=0.85)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("Delta PR-AUC por permutacion")
ax.set_title("Importancia de permutacion — RF residualizado")
plt.tight_layout()
plt.savefig("figures/fig_06a_perm_imp_full.png", dpi=150, bbox_inches="tight")
plt.show()

# Eliminacion hacia atras
elim_order = imp_df.sort_values("imp_mean", ascending=True)["feature"].tolist()
current = MODEL_RES.copy(); history = []
m0, s0 = cv_metrics(current, X_res, y, CV_SEL)
history.append({"n": len(current), "features": current.copy(),
                "removed": "baseline", "mean": m0, "std": s0})
print(f"  {len(current):2d} features [baseline] PR-AUC={m0:.3f}+/-{s0:.3f}")

for feat in elim_order:
    if feat not in current or len(current) <= 1: break
    test_f = [f for f in current if f != feat]
    m, s = cv_metrics(test_f, X_res, y, CV_SEL)
    delta = m - m0
    history.append({"n": len(test_f), "features": test_f.copy(),
                    "removed": feat, "mean": m, "std": s})
    print(f"  {len(test_f):2d} features [-{feat:25s}] PR-AUC={m:.3f}+/-{s:.3f}  D={delta:+.3f}")
    current = test_f

hist_df = pd.DataFrame(history)
best_mu  = hist_df["mean"].max()
best_sd  = hist_df.loc[hist_df["mean"].idxmax(), "std"]
threshold = best_mu - best_sd
parsimonious = hist_df[hist_df["mean"] >= threshold].sort_values("n").iloc[0]
SELECTED = list(parsimonious.features)
print(f"
Features seleccionados ({len(SELECTED)}): {SELECTED}")


In [ ]:
Xs = X_res[SELECTED].copy()
det = cv_metrics(SELECTED, X_res, y, CV_OUTER, n_estimators=700, detailed=True)
m = det.mean(); s = det.std()

print("Metricas CV por animal (5x10, RF residualizado):")
print(f"  PR-AUC = {m.prauc:.3f} +/- {s.prauc:.3f}   (base = {y.mean():.3f})")
print(f"  ROC    = {m.roc:.3f}   +/- {s.roc:.3f}")
print(f"  Brier  = {m.brier:.3f}  +/- {s.brier:.3f}")
print(f"  MCC    = {m.mcc:.3f}   +/- {s.mcc:.3f}")
print(f"  Sens   = {m.sens:.2f}   +/- {s.sens:.2f}")
print(f"  Spec   = {m.spec:.2f}   +/- {s.spec:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, title in zip(axes, ["prauc", "roc", "mcc"], ["PR-AUC", "ROC-AUC", "MCC"]):
    ax.hist(det[col].dropna(), bins=15, color=PALETTE[0], alpha=0.8, edgecolor="white")
    ax.axvline(det[col].mean(), color=PALETTE[1], lw=2, label=f"Media={det[col].mean():.3f}")
    if col == "prauc":
        ax.axvline(y.mean(), color="grey", ls="--", lw=1.5, label=f"Base={y.mean():.3f}")
    ax.set_xlabel(title); ax.set_title(f"Dist. {title}"); ax.legend(fontsize=8)
plt.suptitle("RF residualizado — Metricas CV por animal (5x10)", y=1.02)
plt.tight_layout()
plt.savefig("figures/fig_06a_metricas.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
FINAL = make_pipe_res(SELECTED, n_estimators=1000).fit(Xs, y)
prep  = FINAL.named_steps["prep"]
clf   = FINAL.named_steps["clf"]
Xt    = prep.transform(Xs)
names = list(prep.get_feature_names_out())

explainer = shap.TreeExplainer(clf)
sv_raw    = explainer.shap_values(Xt)
if isinstance(sv_raw, list):
    sv = sv_raw[1]
elif np.asarray(sv_raw).ndim == 3:
    sv = np.asarray(sv_raw)[:, :, 1]
else:
    sv = np.asarray(sv_raw)
exp_val = (float(explainer.expected_value[1])
           if isinstance(explainer.expected_value, (list, np.ndarray))
           else float(explainer.expected_value))

fig, axes = plt.subplots(1, 2, figsize=(14, max(4, 0.5 * len(names))))
plt.sca(axes[0])
shap.summary_plot(sv, Xt, feature_names=names, show=False, max_display=len(names))
axes[0].set_title("SHAP summary — RF residualizado")

mean_abs = np.abs(sv).mean(0)
si = pd.DataFrame({"feature": names, "mean_abs": mean_abs}).sort_values("mean_abs")
axes[1].barh(si.feature, si.mean_abs, color=PALETTE[0], alpha=0.85)
axes[1].set_xlabel("Media |SHAP|")
axes[1].set_title("Importancia SHAP (RF residualizado)")
plt.tight_layout()
plt.savefig("figures/fig_06a_shap.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ALE para los residuales seleccionados
CONT = [f for f in SELECTED if f in NUM_RES and Xs[f].nunique() > 5]
if not CONT:
    CONT = [f for f in SELECTED if f in NUM_RES]

def ale_1d(model, X, feat, bins=10):
    x = X[feat].dropna().values
    q = np.unique(np.quantile(x, np.linspace(0, 1, bins + 1)))
    if len(q) < 3: return np.array([]), np.array([])
    eff, Xc = [], X.copy()
    for i in range(len(q) - 1):
        mask = (X[feat] >= q[i]) & (X[feat] <= q[i + 1])
        if not mask.any(): eff.append(0.0); continue
        lo = Xc.loc[mask].copy(); lo[feat] = q[i]
        hi = Xc.loc[mask].copy(); hi[feat] = q[i + 1]
        eff.append((model.predict_proba(hi)[:, 1]
                    - model.predict_proba(lo)[:, 1]).mean())
    ale = np.cumsum(eff); ale -= ale.mean()
    return (q[:-1] + q[1:]) / 2, ale

if CONT:
    n_c = len(CONT)
    fig, axes_ale = plt.subplots(1, n_c, figsize=(5 * n_c, 4))
    if n_c == 1: axes_ale = [axes_ale]
    for ax, feat in zip(axes_ale, CONT):
        cx, ale_v = ale_1d(FINAL, Xs, feat)
        if not len(cx): continue
        ax.plot(cx, ale_v, "o-", color=PALETTE[1], lw=2, ms=5)
        ax.axhline(0, color="grey", lw=0.7, ls="--")
        ax.fill_between(cx, 0, ale_v, where=(ale_v > 0),
                        alpha=0.18, color=PALETTE[0], label="mayor riesgo TB")
        ax.fill_between(cx, 0, ale_v, where=(ale_v < 0),
                        alpha=0.18, color=PALETTE[1], label="menor riesgo TB")
        ax.set_title(f"ALE: {feat}")
        ax.set_xlabel(feat + "\n(residual intra-finca)")
        ax.legend(fontsize=8)
    plt.suptitle("ALE — efecto intra-finca de VitD/Calcio residual", y=1.02)
    plt.tight_layout()
    plt.savefig("figures/fig_06a_ale.png", dpi=150, bbox_inches="tight")
    plt.show()


## 8. Interpretacion comparativa

| Resultado | Interpretacion |
|---|---|
| PR-AUC(06a) ~ PR-AUC(02c) | La senal de VitD/Calcio es individual — no es la finca disfrazada |
| PR-AUC(06a) << PR-AUC(02c) | La mayor parte de la senal en 02c era efecto ecologico (finca) |
| ALE(VitD_res) decreciente | Mas VitD que la media de la finca → menos TB (efecto intra-finca) |
| ALE(VitD_res) plano | No hay efecto individual de VitD una vez controlada la finca |

> Comparar PR-AUC(06a) con Nb 02c (VitD/Calcio crudos, sin control de finca)
> y con Nb 03c (VitD/Calcio crudos + finca como feature explicita).
